In [2]:
import pandas as pd

df = pd.read_excel("ADEI.xlsx")
df.head()

,ticket_id,opened_at,channel,category,status,resolved_at,csat
0,TK-1001,2026-10-09 17:47,Email,Access,Resolved,2026-10-09 19:49,NaN
1,TK-1002,2026-10-07 09:16,Web,Billing,Resolved,2026-10-07 12:17,5.0
2,TK-1003,2026-10-05 15:14,whatsapp,access,Resolved,2026-10-05 17:36,2.0
3,TK-1004,2026-10-07 14:11,Web,Access,Resolved,2026-10-07 15:50,4.0
4,TK-1005,2026-10-09 12:59,Email,General,Resolved,2026-10-09 18:26,NaN


In [3]:
import pandas as pd
import numpy as np


In [4]:
# Keep a copy of the loaded data and record the input row count
df = df.copy()
original_count = len(df)


In [5]:
# Clean column names and check for required columns
df.columns = df.columns.str.strip().str.lower()

required_columns = {
    "ticket_id", "opened_at", "channel", "category",
    "status", "resolved_at", "csat"
}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

In [6]:
# Clean whitespace and blank text cells
for column in df.select_dtypes(include=["object", "string"]).columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.replace("\u00a0", " ", regex=False)
        .str.strip()
        .replace("", pd.NA)
    )


In [7]:
# Standardise ticket IDs
df["ticket_id"] = df["ticket_id"].astype("string").str.upper()

# Standardise common channel labels
channel_mapping = {
    "whatsapp": "WhatsApp",
    "wa": "WhatsApp",
    "whats app": "WhatsApp",
    "web": "Web",
    "website": "Web",
    "email": "Email",
    "sms": "SMS",
}

df["channel"] = (
    df["channel"].astype("string").str.lower()
    .map(channel_mapping)
    .fillna(df["channel"])
)

In [8]:

# Standardise category and status capitalisation
for column in ["category", "status"]:
    df[column] = df[column].astype("string").str.title()


In [9]:
# Convert timestamps; unparseable values become missing and are flagged
df["opened_at"] = pd.to_datetime(df["opened_at"], errors="coerce")
df["resolved_at"] = pd.to_datetime(df["resolved_at"], errors="coerce")

df["invalid_opened_at"] = df["opened_at"].isna()
df["invalid_resolved_at"] = (
    df["status"].eq("Resolved") & df["resolved_at"].isna()
)
df["unexpected_resolved_at"] = (
    df["status"].ne("Resolved") & df["resolved_at"].notna()
)

In [10]:
# Remove exact duplicate rows, keeping the first copy
data_columns = [
    column for column in df.columns
    if column not in {
        "invalid_opened_at",
        "invalid_resolved_at",
        "unexpected_resolved_at",
    }
]
exact_duplicate_mask = df.duplicated(subset=data_columns, keep="first")
exact_duplicates = df.loc[exact_duplicate_mask].copy()
df = df.loc[~exact_duplicate_mask].copy()
after_exact_deduplication = len(df)

In [11]:
# Flag all rows with a repeated non-missing ticket ID
duplicate_id_mask = (
    df["ticket_id"].notna()
    & df["ticket_id"].duplicated(keep=False)
)
conflicting_records = df.loc[duplicate_id_mask].copy()
df["conflicting_id"] = duplicate_id_mask

In [12]:
# Flag missing IDs and labels; preserve missing channel/category as Unknown
df["missing_ticket_id"] = df["ticket_id"].isna()
df["missing_channel"] = df["channel"].isna()
df["missing_category"] = df["category"].isna()

df["channel"] = df["channel"].fillna("Unknown")
df["category"] = df["category"].fillna("Unknown")

In [13]:
# Convert optional CSAT values to numbers; scores must be from 1 to 5
df["csat"] = pd.to_numeric(df["csat"], errors="coerce")
df["invalid_csat"] = df["csat"].notna() & ~df["csat"].between(1, 5)
df.loc[df["invalid_csat"], "csat"] = np.nan

In [14]:
# Calculate resolution time only when both timestamps are available
resolved_mask = (
    df["status"].eq("Resolved")
    & df["opened_at"].notna()
    & df["resolved_at"].notna()
)

df["resolution_minutes"] = np.nan
df.loc[resolved_mask, "resolution_minutes"] = (
    df.loc[resolved_mask, "resolved_at"]
    - df.loc[resolved_mask, "opened_at"]
).dt.total_seconds() / 60

In [15]:
# Flag negative durations and leave them out of the duration metric
df["negative_duration"] = df["resolution_minutes"] < 0
df.loc[df["negative_duration"], "resolution_minutes"] = np.nan

In [16]:
# Record why each row is excluded
df["exclusion_reason"] = ""

exclusion_rules = [
    (df["missing_ticket_id"], "Missing ticket ID"),
    (df["conflicting_id"], "Repeated/conflicting ticket ID"),
    (df["invalid_opened_at"], "Invalid or missing opening timestamp"),
]

for mask, reason in exclusion_rules:
    df.loc[mask, "exclusion_reason"] += reason + "; "

excluded_records = df[df["exclusion_reason"] != ""].copy()
cleaned_df = df[df["exclusion_reason"] == ""].copy()

In [17]:

# Validation checks
assert cleaned_df["ticket_id"].notna().all(), "Missing ticket IDs remain."
assert cleaned_df["ticket_id"].is_unique, "Repeated ticket IDs remain."
assert cleaned_df["opened_at"].notna().all(), "Invalid opening timestamps remain."
assert (
    cleaned_df.loc[
        cleaned_df["status"] != "Resolved", "resolution_minutes"
    ].isna().all()
), "Unresolved tickets have resolution durations."
assert cleaned_df["csat"].dropna().between(1, 5).all(), "Invalid CSAT remains."

print("Validation checks passed.")
print("Input rows:", original_count)
print("Exact duplicate rows removed:", len(exact_duplicates))
print("Rows after exact duplicate removal:", after_exact_deduplication)
print("Rows excluded:", len(excluded_records))
print("Cleaned output rows:", len(cleaned_df))
print("Repeated/conflicting ID rows:", int(df["conflicting_id"].sum()))
print("Invalid opening timestamps:", int(df["invalid_opened_at"].sum()))
print("Negative durations flagged:", int(df["negative_duration"].sum()))
print("Invalid CSAT scores set to missing:", int(df["invalid_csat"].sum()))

# Save results in a separate workbook
summary = pd.DataFrame({
    "Measure": [
        "Input rows",
        "Exact duplicate rows removed",
        "Rows after exact duplicate removal",
        "Rows excluded",
        "Cleaned output rows",
        "Repeated/conflicting ID rows",
        "Invalid opening timestamps",
        "Negative durations flagged",
        "Invalid CSAT scores set to missing",
    ],
    "Count": [
        original_count,
        len(exact_duplicates),
        after_exact_deduplication,
        len(excluded_records),
        len(cleaned_df),
        int(df["conflicting_id"].sum()),
        int(df["invalid_opened_at"].sum()),
        int(df["negative_duration"].sum()),
        int(df["invalid_csat"].sum()),
    ],
})

with pd.ExcelWriter("ADEI_Cleaning_Results.xlsx", engine="openpyxl") as writer:
    cleaned_df.to_excel(writer, sheet_name="Cleaned_Tickets", index=False)
    excluded_records.to_excel(writer, sheet_name="Excluded_Records", index=False)
    exact_duplicates.to_excel(writer, sheet_name="Exact_Duplicates", index=False)
    conflicting_records.to_excel(writer, sheet_name="Conflicting_IDs", index=False)
    summary.to_excel(writer, sheet_name="Cleaning_Summary", index=False)

display(summary)

Validation checks passed.
Input rows: 146
Exact duplicate rows removed: 1
Rows after exact duplicate removal: 145
Rows excluded: 6
Cleaned output rows: 139
Repeated/conflicting ID rows: 2
Invalid opening timestamps: 3
Negative durations flagged: 1
Invalid CSAT scores set to missing: 2


,Measure,Count
0,Input rows,146
1,Exact duplicate rows removed,1
2,Rows after exact duplicate removal,145
3,Rows excluded,6
4,Cleaned output rows,139
5,Repeated/conflicting ID rows,2
6,Invalid opening timestamps,3
7,Negative durations flagged,1
8,Invalid CSAT scores set to missing,2
